# Emina & Kropff (2026): Prospective Coding and Path Integration

> **一句话概括**：一个带有 firing-rate adaptation（放电率适应）和 global divisive inhibition（全局除法抑制）的竞争网络，在平移不变的移动 tutor 输入下，通过局部 Hebbian plasticity（赫布可塑性）从随机权重中学习出 Gaussian feedforward / recurrent connectivity；适应导致活动 bump 向前偏移，而学得的 recurrent connectivity 允许 bump 在撤去位置输入后自行运动，再由全局 speed current（速度电流）调节其运动速度，实现 **unidirectional path integration（单向路径积分）**。

这篇文章和 Vafidis 的共同目标都是解释“精细的连续吸引子连接如何通过学习形成”，但两者的核心机制不同：

- **Vafidis**：视觉信号作为 teacher，通过双室神经元的 prediction error 学习 recurrent / rotation weights。
- **Emina & Kropff**：没有显式 teacher-error；由 moving tutor 引发的 pre-post 共激活、weight-dependent decay、适应和全局竞争共同塑造连接。
- **本文最特别的结果**：prospective coding 不要求 recurrent excitation；单纯 feedforward hierarchy 也能逐层放大预测性位移。

> 下面统一采用便于 coding 的矩阵约定：$J_{ij}$ 表示 input neuron $j\to$ competitive neuron $i$，$W_{ij}$ 表示 competitive neuron $j\to i$。原文 Eq. 6 与 Eq. 10 附近的文字对 $x,x^{\prime}$ 的 pre/post 指派有轻微互换；由于最终平衡解只依赖 $x-x^{\prime}$，不影响 Gaussian 结论，但实现时必须固定一种矩阵方向。

# 1. 网络对象与离散化后的数据维度

## 1.1 空间与神经元

- 一维空间坐标：$x\in[0,L)$
    - 原文把 $x$ 同时用作 neuron label（神经元在内部流形上的坐标）和 tuning center（调谐中心）。
    - tutor 从 $0$ 单向移动到 $L$ 后重新从 $0$ 开始。
    - 假设 $L\gg \sigma_R$，从而忽略边界并把积分近似为 $\mathbb{R}$ 上的积分。
- input/tutor layer（输入/引导层）神经元数：$N_{\mathrm{in}}$
    - 神经元密度：$\rho_{\mathrm{in}}=N_{\mathrm{in}}/L$
    - 离散 preferred positions：$\{x^{\mathrm{in}}_j\}_{j=1}^{N_{\mathrm{in}}}$
- competitive layer（竞争层）神经元数：$N_c$
    - 神经元密度：$\rho_c=N_c/L$
    - 离散 internal coordinates：$\{x^c_i\}_{i=1}^{N_c}$

## 1.2 状态变量

- tutor activity（引导层放电率）：
  $\vec R(t)\in\mathbb{R}^{N_{\mathrm{in}}}$，第 $j$ 个分量为 $R_j(t)$
- competitive membrane potential（竞争层膜电位）：
  $\vec U(t)\in\mathbb{R}^{N_c}$
- firing-rate adaptation variable（放电率适应变量）：
  $\vec V(t)\in\mathbb{R}^{N_c}$
- competitive firing rate（竞争层放电率）：
  $\vec r(t)\in\mathbb{R}^{N_c}$
- total synaptic current（总突触电流）：
  $\vec I(t)\in\mathbb{R}^{N_c}$

## 1.3 权重矩阵

- feedforward weight（前馈权重）：
  $\mathbf J(t)\in\mathbb{R}_{+}^{N_c\times N_{\mathrm{in}}}$
    - $J_{ij}$：input neuron $j\to$ competitive neuron $i$
    - 全部为 excitatory；更新后若小于 $0$，simulation 中 clip 到 $0$
- recurrent weight（递归权重）：
  $\mathbf W(t)\in\mathbb{R}_{+}^{N_c\times N_c}$
    - $W_{ij}$：competitive neuron $j\to i$
    - 也是 excitatory；全局抑制不是由 $\mathbf W$ 中的负权重实现，而是由 divisive normalization 实现

> 连续形式中的 $\begin{aligned}\rho\int \mathrm{d}x^{\prime}\end{aligned}$ 对均匀离散网格等价于 $\begin{aligned}\rho\Delta x\sum_j\end{aligned}$。由于 $\begin{aligned}\Delta x=\frac{L}{N}\end{aligned}$，有 $\rho\Delta x=1$，因此代码中通常可以直接写成矩阵乘法 $\mathbf J\vec R$ 或 $\mathbf W\vec r$。

# 2. Moving tutor input（移动引导输入）

真实/外部 stimulus position（刺激位置）为

$$
z(t)=vt\pmod L,
$$

其中 $v>0$ 是训练阶段的恒定单向速度。

第 $j$ 个 input neuron 的活动为 normalized Gaussian tuning curve（归一化高斯调谐曲线）：

$$
R_j(t)
=
A_R\,\mathcal N\!\left(x^{\mathrm{in}}_j;\,z(t),\sigma_R\right),
$$

连续写法为

$$
\vec{R}(x,t)=A_R\mathcal N(x;vt,\sigma_R) = A_{R} \frac{1}{\sqrt{2\pi\sigma_R^2}}\exp\left[-\frac{(x-vt)^2}{2\sigma_R^2}\right],
$$

> $\begin{aligned}\mathcal N(x;\mu,\sigma)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\left[-\frac{(x-\mu)^2}{2\sigma^2}\right]\end{aligned}$. 

- $\vec R(t)\in\mathbb{R}^{N_{\mathrm{in}}}$
- $A_R$：整个群体 profile 的 integrated drive（积分总活动），不是 peak height
- $\sigma_R>0$：输入调谐宽度
- $v>0$：单向移动速度
- $L\gg\sigma_R$：理论分析忽略两端边界的条件

实现时有两种选择：

1. **忠实于原文**：线段上移动，到 $L$ 时 reset，并只在远离边界处取统计量。
2. **数值上更干净**：使用 periodic distance 构造环形 Gaussian；这样会把模型轻微改写为 ring，但能避免 reset discontinuity。

# 3. Competitive layer dynamics（竞争层动力学）

每个 competitive neuron 有两个动力学变量：膜电位 $U$ 和适应变量 $V$。

连续形式：

$$
\begin{aligned}\tau\frac{\partial U(x,t)}{\partial t}&=-U(x,t)-V(x,t)+I(x,t),\\
\tau_v\frac{\partial V(x,t)}{\partial t}&=-V(x,t)+mU(x,t).\end{aligned}
$$

离散矢量形式：

$$
\begin{aligned}
\tau\frac{\mathrm{d}\vec U}{\mathrm{d}t} &=-\vec U-\vec V+\vec I,\\
\tau_v\frac{\mathrm{d}\vec V}{\mathrm{d}t}&=-\vec V+m\vec U.
\end{aligned}
$$

- $\tau>0$：membrane time constant（膜电位时间常数）
- $\tau_v>0$：adaptation time constant（适应时间常数），通常 $\tau_v\gg\tau$
- $m\ge 0$：adaptation strength（适应强度）
- $-\vec V$：对近期活动的延迟负反馈
- 在移动 bump 中，$V$ 的 profile 落后于 $U$；落后的适应会抑制 bump 后方并释放前方神经元，产生前移倾向

仅含 feedforward input 时：

$$
I_{\mathrm{ff}}(x,t)
=
\rho_{\mathrm{in}}\int \mathrm{d}x^{\prime}\,J(x,x^{\prime})R(x^{\prime},t),
$$

离散形式为

$$
\vec I_{\mathrm{ff}}(t)=\mathbf J(t)\vec R(t).
$$

# 4. Firing rate 与 global divisive inhibition

competitive firing rate 为 quadratic rectification（平方整流）加全局除法归一化：

$$
r(x,t)=\frac{[U(x,t)]_+^2}{B(t)},\quad B(t)=1+k\rho_c\int \mathrm{d}x^{\prime}\,[U(x^{\prime},t)]_+^2.
$$

离散形式：

$$
r_i(t)
=
\frac{[U_i(t)]_+^2}
{1+k\sum_{\ell=1}^{N_c}[U_\ell(t)]_+^2}.
$$

- $\vec r(t)\in\mathbb{R}_{\ge0}^{N_c}$
- $k>0$：global inhibition strength（全局抑制强度）
- $B(t)$ 是所有神经元共享的 normalization factor
- 该机制相当于 **local excitation + global competition**：
    - $\mathbf J,\mathbf W$ 只含 excitatory weights
    - 抑制通过 population-level denominator 施加
- quadratic nonlinearity 是后续 path-integration 机制的关键：uniform baseline current 会改变 bump 的有效宽度，进而改变运动速度

> 这里的 global inhibition 不是一个显式 inhibitory population，因此模型在生物学上是 effective rate model（有效放电率模型）。

# 5. Feedforward Hebbian learning rule（前馈赫布学习规则）

对 plastic synapse $j\to i$：

$$
\frac{\mathrm{d}J_{ij}}{\mathrm{d}t}
=
\eta_J\,r_i(t)
\left[
R_j(t)-\alpha_J J_{ij}(t)^\beta
\right].
$$

矩阵逐元素写法：

$$
\frac{\mathrm{d}\mathbf J}{\mathrm{d}t}
=
\eta_J
\left[
\vec r\,\vec R^{\top}
-
\alpha_J\,
\vec r\,\vec 1_{N_{\mathrm{in}}}^{\top}
\odot
\mathbf J^{\odot\beta}
\right].
$$

- $\eta_J>0$：feedforward learning rate
- $\alpha_J>0$：weight-dependent decay strength
- $\beta>0$：decay exponent
- $\odot$：elementwise product
- $\mathbf J^{\odot\beta}$：逐元素幂
- timescale separation（时间尺度分离）：
  $1/\eta_J\gg\tau_v\gg\tau$

对单个突触的解释：

$$
\underbrace{\eta_J r_iR_j}_{\text{Hebbian potentiation}}
-
\underbrace{\eta_J\alpha_Jr_iJ_{ij}^{\beta}}_{\text{activity-gated local decay}}.
$$

这个规则只需要 synapse-local quantities：

- presynaptic activity $R_j$
- postsynaptic activity $r_i$
- 当前 weight $J_{ij}$

它不是 Vafidis 式的 prediction-error rule；没有
$f(V_a)-f(pV_d)$ 这一显式误差信号。

# 6. Feedforward weights 的 equilibrium solution（平衡解）

由于 learning 比 neural dynamics 慢，可以对完整 stimulus cycle 中的 bump position $z$ 做平均：

$$
0
=
\left\langle
r_i(z)\left[R_j(z)-\alpha_JJ_{ij}^{\beta}\right]
\right\rangle_z.
$$

因此

$$
J_{ij}^{\mathrm{eq}}
=
\left[
\frac{\langle r_i(z)R_j(z)\rangle_z}
{\alpha_J\langle r_i(z)\rangle_z}
\right]^{1/\beta}.
$$

若 tutor activity 与 competitive activity 都是 Gaussian profile，则平衡权重也为 Gaussian：

$$
J^{\mathrm{eq}}(x,x^{\prime})
=
A_J\mathcal N(x;x^{\prime},\sigma_J).
$$

网络活动采用 Gaussian ansatz：

$$
U^{\mathrm{eq}}(x,z)
=
A_u\mathcal N(x;z,\sigma_u),
$$

$$
V^{\mathrm{eq}}(x,z)
=
A_v\mathcal N(x;z-d,\sigma_u),
$$

$$
r^{\mathrm{eq}}(x,z)
=
A_r\mathcal N\left(x;z,\frac{\sigma_u}{\sqrt2}\right).
$$

其中 $d>0$ 表示 adaptation profile 落后于 $U$ bump。

平衡连接宽度与活动宽度为

$$
\sigma_J
=
\sqrt{\frac{3\beta}{2-\beta}}\,\sigma_R,
\qquad
\sigma_u
=
\sqrt{\frac{2\beta+2}{2-\beta}}\,\sigma_R,
$$

要求

$$
0<\beta<2.
$$

特别地，$\beta=0.5$ 时：

$$
\sigma_J=\sigma_R,
\qquad
\frac{\sigma_u}{\sqrt2}=\sigma_R.
$$

即 feedforward weight、input firing profile 与 competitive firing profile 具有匹配的空间尺度。这也是原文加入 recurrent learning 时固定使用 $\beta=0.5$ 的原因。

权重幅值为

$$
A_J
=
\left(\frac{A_R}{\alpha_JC_\beta}\right)^{1/\beta},
\qquad
C_\beta
=
\sqrt{\frac{(2\pi\sigma_J^2)^{1-\beta}}{\beta}}.
$$

直观上：

- $\beta$ 控制“学得的空间尺度”
- $\alpha_J$ 主要反向控制权重幅值
- translationally invariant experience 使每个位置经历同样的统计量，因此最终 $J(x,x^{\prime})$ 只依赖 $x-x^{\prime}$

# 7. Feedforward connectivity 的稳定性

对平衡权重施加小扰动

$$
J(x,x^{\prime},t)=J^{\mathrm{eq}}(x,x^{\prime})+\delta J(x,x^{\prime},t).
$$

线性化后得到

$$
\tau_J(x,x^{\prime})\frac{\partial}{\partial t}\delta J(x,x^{\prime},t) = \int \mathrm{d}x^{\prime\prime}\,K(x,x^{\prime},x^{\prime\prime})\delta J(x^{\prime\prime},x^{\prime},t)
-
\delta J(x,x^{\prime},t).
$$

因为平衡解只依赖 relative displacement（相对位移）

$$
s=x-x^{\prime},
$$

可以把问题化为一维 integral operator（积分算符）的广义本征值问题：

$$
\hat L[f_n(s)]
=
\lambda_n\tau_J(s)f_n(s).
$$

稳定条件为

$$
\max_n\operatorname{Re}\lambda_n<0.
$$

原文 Fig. 2 的数值谱分析表明，在所扫描的 input density $\rho_{\mathrm{in}}$、decay strength $\alpha_J$ 与多个 $\beta$ 范围内，leading eigenvalue 保持负实部，因此学得的 Gaussian feedforward profile 不是偶然的瞬时结构，而是 learning dynamics 的稳定固定点。

> 这一部分分析的是 **weight-learning dynamics 的稳定性**，不是神经活动 Jacobian 的稳定性。它和 Clark 中对 neural-state Jacobian / recurrent spectrum 的分析对象不同。

# 8. Prospective coding（前瞻编码）的来源

Gaussian zeroth mode 描述对称 bump；first Hermite mode（第一阶 Hermite 模式）描述左右不对称。原文使用

$$
U_\gamma^{\mathrm{eq}}(x,z)
=
A_u\mathcal N(x;z,\sigma_u)
\left[
1+\gamma\frac{x-z}{\sigma_u}
\right].
$$

当 $|\gamma|\ll1$ 时，

$$
\mathcal N(x;z+\sigma_u\gamma,\sigma_u)
\approx
\mathcal N(x;z,\sigma_u)
\left[
1+\gamma\frac{x-z}{\sigma_u}
\right].
$$

因此 bump center 的位移近似为

$$
\Delta z_{\mathrm{pred}}
=
\sigma_u\gamma.
$$

- $\gamma>0$：bump 位于当前 tutor 的前方，prospective coding
- $\gamma=0$：无预测性位移
- $\gamma<0$：bump 落后于 tutor，retrospective coding
- 对 $v>0$，anticipation time 可写为
  $T_{\mathrm{ant}}\approx\sigma_u\gamma/v$

定义无量纲变量

$$
u=\frac{\tau v}{\sqrt2\sigma_u},
\qquad
y=\frac{\mathrm{d}}{\sqrt2\sigma_u},
\qquad
\Gamma=\frac{\tau_v}{\tau}.
$$

投影到前两个 Hermite modes 后，稳态解满足

$$
\gamma
=
\frac{my}{\Gamma yu+1}-u,
$$

以及

$$
y
=
\frac{(m+1)-\Gamma u^2}{2\Gamma u}
\left[
\sqrt{
1+
\frac{4u^2\Gamma(\Gamma+1)}
{[\Gamma u^2-(m+1)]^2}
}
-1
\right].
$$

机制解释：

1. moving input 把活动 bump 向前拖动；
2. adaptation $V$ 落后于 $U$；
3. bump 后方受到更强抑制并衰减；
4. global inhibition 下降后，前方尚未活动的神经元更容易被招募；
5. 若 adaptation 足够强、速度不过快，center of mass 可领先 tutor。

因此 prospective shift 不是被写进权重中的固定不对称，而是 **对称 Gaussian connectivity + 动态适应** 的结果。

# 9. Multilayer feedforward hierarchy（多层前馈层级）

把多个 competitive layers 串联，每层都使用相同类型的：

- $\tau,\tau_v,m$
- quadratic activation + divisive inhibition
- feedforward Hebbian rule

第 $\ell$ 层把前一层活动作为自己的 tutor，并学习出 Gaussian feedforward weights。每层产生约

$$
\Delta_\ell\approx\sigma_u\gamma
$$

的前向偏移，因此经过 $M$ 层后的累积 shift 近似为

$$
\Delta_{\mathrm{total}}
\approx
M\sigma_u\gamma.
$$

原文 Fig. 4 展示：

- 每一层的随机连接都逐渐与理论 Gaussian profile 高相关；
- activity bump 随层数逐步向未来位置移动；
- shift 与 layer number 近似线性。

但深度不是无限有益：

- 每层都会累积 shape error / approximation error
- 层数过多会导致空间信息退化
- 因此 hierarchy 的作用是 **potentiate（增强）** predictive shift，而不是创造其必要条件；单层调大 $m$ 也可产生 anticipation

这一结果是文章对 entorhinal superficial layers 的主要解释：强 recurrent excitation 并非产生 prospective coding 的必要条件，feedforward transformations 本身也可以贡献预测性位移。

# 10. 加入 recurrent connectivity（递归连接）

总输入变为

$$
I(x,t)=I_{\mathrm{ff}}(x,t)+I_{\mathrm{rec}}(x,t),
$$

$$
I_{\mathrm{rec}}(x,t)
=
\rho_c\int \mathrm{d}x^{\prime}\,W(x,x^{\prime})r(x^{\prime},t).
$$

离散形式：

$$
\vec I(t)
=
\mathbf J(t)\vec R(t)
+
\mathbf W(t)\vec r(t).
$$

recurrent learning rule 为

$$
\frac{\mathrm{d}W_{ij}}{\mathrm{d}t}
=
\eta_W
\left[
r_i(t)r_j(t)
-
\alpha_Wr_j(t)W_{ij}(t)^\beta
\right],
$$

也可写成

$$
\frac{\mathrm{d}W_{ij}}{\mathrm{d}t}
=
\eta_Wr_j(t)
\left[
r_i(t)-\alpha_WW_{ij}(t)^\beta
\right].
$$

- $\eta_W>0$：recurrent learning rate
- $\alpha_W>0$：recurrent decay strength
- $\beta=0.5$：原文 recurrent experiments 使用的值
- $W_{ij}\ge0$：excitatory recurrent connection
- 对称移动经验使平衡 recurrent profile 近似为

$$
W^{\mathrm{eq}}(x,x^{\prime})
=
A_W\mathcal N(x;x^{\prime},\sigma_W),
$$

且在 $\beta=0.5$ 时

$$
\sigma_W\approx\sigma_R.
$$

注意：这个规则没有直接要求 $W_{ij}=W_{ji}$，但 translationally invariant、均匀遍历的训练统计会使平均连接 profile 近似对称 Gaussian。

原文的数值 phase diagram（Fig. 6）表明：

- 需要 tutor/feedforward drive 足够强，才能先建立空间编码；
- 大致需要 $\alpha_W\gtrsim\alpha_J$，避免 recurrent signal 在学习早期压过 tutor；
- recurrent drive 也不能弱到完全学不出来；
- tutor 太强同样会破坏“慢学习、快动力学”的 averaging assumption。

# 11. 撤去 tutor 后的 self-sustained moving bump

学得 $\mathbf W$ 后，令

$$
I_{\mathrm{ff}}=0.
$$

local recurrent excitation、global inhibition 与 firing-rate adaptation 共同支持一个自行传播的 Gaussian bump。其理论 intrinsic speed（内禀速度）为

$$
v_{\mathrm{int}}(m)
=
\frac{\sqrt2\sigma_u}{\tau_v}
\sqrt{
\frac{m\tau_v}{\tau}
-
\sqrt{\frac{m\tau_v}{\tau}}
}.
$$

令

$$
q=\frac{m\tau_v}{\tau},
$$

则实数 moving solution 要求大致为

$$
q>1.
$$

原文指出 projection-method Gaussian ansatz 系统性高估实际速度；跨参数 simulation 得到经验校正系数约 $0.72$：

$$
v_{\mathrm{int}}^{\mathrm{corr}}
\approx
0.72\,
\frac{\sqrt2\sigma_u}{\tau_v}
\sqrt{q-\sqrt q}.
$$

这是一个重要实现细节：

- 不乘 $0.72$ 时，理论预测与 simulation 会有稳定比例偏差；
- 作者认为偏差可能来自 $V(x,z)$ 并非严格 Gaussian；
- $m$ 决定 spontaneous motion speed，而不是直接由外部 velocity input 决定；
- symmetric recurrent weights 本身维持 bump，adaptation 将其从 static bump 推入 moving-bump regime。

> 这仍不是完整的任意方向路径积分器。文章只处理一维、单向运动；方向不是由一个正负 velocity channel 显式控制的。

# 12. Speed current 与 unidirectional path integration

仅靠固定 $m,\tau,\tau_v,\sigma_u$，bump 只能以固定的 $v_{\mathrm{int}}$ 运动。为了追踪 time-varying external speed $v(t)$，作者加入 spatially uniform background current（空间均匀背景电流）：

$$
\vec I(t)
=
\mathbf W\vec r(t)
+
I_{\mathrm{speed}}(t)\vec 1_{N_c},
$$

$$
I_{\mathrm{speed}}(t)
=
g\left[
v(t)-v_{\mathrm{int}}(m)
\right].
$$

其中测试时撤去位置 tutor：

$$
\mathbf J\vec R=0.
$$

作者使用 baseline-shift ansatz：

$$
U_\epsilon^{\mathrm{eq}}(x,z)
=
U^{\mathrm{eq}}(x,z)
+
\epsilon_u I_{\mathrm{speed}},
$$

$$
V_\epsilon^{\mathrm{eq}}(x,z)
=
V^{\mathrm{eq}}(x,z)
+
\epsilon_v I_{\mathrm{speed}},
$$

并有

$$
\epsilon_u+\epsilon_v=1,
\qquad
\epsilon_v=m\epsilon_u,
$$

因此

$$
\epsilon_u=\frac{1}{1+m},
\qquad
\epsilon_v=\frac{m}{1+m}.
$$

由于 firing rate 为 $[U]_+^2/B$，uniform baseline shift 会产生与原 Gaussian 不同宽度的交叉项，使 bump wi\mathrm{d}th 近似变为

$$
\operatorname{std}_x[U]
\approx
\sigma_u
\left[
1+
\frac{\sqrt\pi\sigma_u}{A_u}
\epsilon_u I_{\mathrm{speed}}
\right].
$$

而 moving-bump speed 与 wi\mathrm{d}th 成正比，因此

$$
\frac{\mathrm{d}z}{\mathrm{d}t}
\approx
v_{\mathrm{int}}(m)
\left[
1+
\frac{\sqrt\pi\sigma_u}{A_u}
\epsilon_u I_{\mathrm{speed}}
\right].
$$

数据流为：

$$
v(t)
\rightarrow
I_{\mathrm{speed}}(t)
\rightarrow
\text{uniform baseline shift}
\rightarrow
\text{bump width change}
\rightarrow
\text{bump speed change}
\rightarrow
z_{\mathrm{net}}(t)\approx\int v(t)\,\mathrm{d}t.
$$

适用范围与限制：

- 需要 $|\epsilon_uI_{\mathrm{speed}}|$ 相对 bump amplitude 足够小，才能使用一阶线性化；
- 高正电流处理论与 simulation 偏差增大；
- 该机制只调节既有运动方向上的速度，故称 **unidirectional path integration**；
- 没有显式 left/right velocity populations，不能直接处理速度符号翻转或二维方向向量。

# 13. End-to-end data flow（从训练到测试）

## 13.1 Stage A：只学习 feedforward representation

每个 neural time step：

1. 更新 tutor position：
   $z(t+\Delta t)=z(t)+v\Delta t\pmod L$
2. 计算 $\vec R(t)$
3. 计算 $\vec I_{\mathrm{ff}}=\mathbf J\vec R$
4. 更新
   $\vec U,\vec V$
5. 计算
   $\vec r=[\vec U]_+^{\odot2}/B$
6. 更新
   $\mathbf J$
7. clip：
   $\mathbf J\leftarrow\max(\mathbf J,0)$

结果：

$$
\mathbf J_{\mathrm{random}}
\longrightarrow
\mathbf J_{\mathrm{Gaussian}},
$$

并形成连续的一维 population code。

## 13.2 Stage B：同时学习 recurrent attractor connectivity

每个 time step：

1. 计算
   $\vec I=\mathbf J\vec R+\mathbf W\vec r$
2. 更新
   $\vec U,\vec V,\vec r$
3. 同时更新
   $\mathbf J,\mathbf W$
4. 继续用 moving tutor 锚定位置

结果：

$$
(\mathbf J,\mathbf W)_{\mathrm{random}}
\longrightarrow
(\mathbf J_{\mathrm{Gaussian}},\mathbf W_{\mathrm{Gaussian}}).
$$

## 13.3 Stage C：测试 autonomous dynamics

- freeze $\mathbf J,\mathbf W$
- 撤去 tutor：
  $\vec R=\vec0$
- 从某个局部 bump initial condition 开始
- 观察 bump 是否持续存在并以
  $v_{\mathrm{int}}(m)$ 传播

## 13.4 Stage D：测试 path integration

- 输入 time-varying $v(t)$
- 计算
  $I_{\mathrm{speed}}=g[v(t)-v_{\mathrm{int}}]$
- 更新 recurrent dynamics
- 解码 bump position，例如 circular/linear center of mass：

$$
\hat z(t)
=
\frac{\sum_i x_i^c r_i(t)}
{\sum_i r_i(t)}
$$

若使用 periodic domain，应改用 complex population vector，避免跨边界时普通质心跳变。

性能指标：

- weight-to-Gaussian correlation
- bump wi\mathrm{d}th 与理论值的误差
- prospective shift $\Delta z$
- autonomous bump speed error
- path-integration position error
  $|\hat z(t)-z_{\mathrm{true}}(t)|$
- speed gain：
  $d\hat z/\mathrm{d}t$ 对 $v(t)$ 的回归斜率

# 14. 最小 toy model 设计建议

## 14.1 建议的最小规模

- $L=100$
- $N_{\mathrm{in}}=N_c=128$ 或 $256$
- $\Delta t=5\times10^{-3}\,\mathrm{s}$
- $\tau=15\,\mathrm{ms}$
- $\tau_v=600\,\mathrm{ms}$
- $\sigma_R=5$
- $A_R=30$
- $\beta=0.5$
- $m\approx0.2$
- $\eta_J,\eta_W\sim5\times10^{-4}$ 至 $10^{-3}$
- 从小的 nonnegative random $\mathbf J,\mathbf W$ 开始

这些值来自原文代表性 simulation，但不同 normalization convention 会显著改变 $k,\alpha_J,\alpha_W$ 的数值，不能机械照搬。

## 14.2 建议分四步复现

1. **固定理论 Gaussian $\mathbf J$**，先验证 $\vec U,\vec V,\vec r$ 能形成 moving bump 与 prospective shift。
2. **关闭 $\mathbf W$，学习 $\mathbf J$**，验证 Eq. 13 的 wi\mathrm{d}th law。
3. **加入 $\mathbf W$ 学习**，画 $(\alpha_J,\alpha_W)$ phase diagram。
4. **冻结权重并撤去 tutor**，再加入 $I_{\mathrm{speed}}$ 测试 path integration。

## 14.3 必须避免的实现错误

- 把 $A_R$ 当成 Gaussian peak，而原文把它定义为 integrated drive。
- 忘记 normalized Gaussian 中的 $1/\sqrt{2\pi\sigma^2}$。
- 把 $\mathbf J$ 的矩阵方向写反。
- 在离散积分中既乘 $\rho$ 又遗漏/重复 $\Delta x$。
- 对 $\mathbf J,\mathbf W$ 使用普通矩阵幂，而不是 elementwise power。
- 没有在每步更新后执行 nonnegative clipping。
- learning rate 太大，破坏 $1/\eta\gg\tau_v$ 的时间尺度分离。
- 用普通 center of mass 解码 periodic bump，导致越界跳变。
- 在还没有稳定 feedforward representation 时就让 recurrent signal 占主导。

# 15. 与 Vafidis 模型的直接对照

| 问题 | Vafidis (2022) | Emina & Kropff (2026) |
|---|---|---|
| 表征变量 | ring 上的 head direction | 一维 segment/manifold 上的位置 |
| 外部 teacher | visual bump 直接驱动 HD proximal compartment | 单向移动 Gaussian tutor layer |
| 神经元机制 | HD 为 two-compartment associative neuron | 单变量膜电位 $U$ + adaptation variable $V$ |
| plasticity signal | distal prediction error $\times$ presynaptic PSP | pre-post coactivation + weight-dependent decay |
| recurrent architecture | HD、left/right HR populations | 单一 competitive population |
| 速度输入 | left/right HR cells 接收 signed angular velocity | uniform scalar current $I_{\mathrm{speed}}$ |
| bump 移动机制 | rotation populations 对 bump 定向推动 | firing-rate adaptation 使 bump 自推进 |
| prospective coding | 不是核心目标 | feedforward layer 中自然出现 |
| path integration | 可处理正负角速度、ring 上 gain-1 | 单向速度调节，当前只是一维 unidirectional PI |
| learned connectivity | recurrent 与 HR-to-HD weights | feedforward 与 recurrent Gaussian weights |
| 理论重点 | biologically local predictive learning | equilibrium Gaussian solution、Hermite projection、weight stability |
| 抑制 | 显式/有效 global inhibition terms | divisive normalization |
| 与 Clark 的潜在连接 | 可分析 mature network neural Jacobian | 可分别分析 weight-learning operator 与 mature recurrent neural Jacobian |

最关键的概念区别：

- Vafidis 学习的是“如何让 distal/recurrent pathway 复现视觉 teacher 所要求的输出”。
- Emina & Kropff 学习的是“在均匀移动经验下，哪些连接是 Hebbian-learning dynamics 的统计平衡解”。
- Vafidis 的运动方向来自 left/right velocity channels。
- Emina & Kropff 的运动首先由 adaptation 自发产生，speed current 只调节其速度。

# 16. 如何阅读原文图

- **Fig. 1**：随机 feedforward weights 如何收敛成 Gaussian；重点检查 Eq. 12–13。
- **Fig. 2**：learning dynamics 的 generalized eigenvalue stability，不是 neural-state Jacobian。
- **Fig. 3**：$\gamma$ 如何随 adaptation strength $m$ 和 tutor speed $v$ 改变。
- **Fig. 4**：多层 feedforward hierarchy 中 prospective shift 近似线性累积。
- **Fig. 5**：feedforward 与 recurrent Gaussian connectivity 可同时学出；撤去 tutor 后仍有 moving bump。
- **Fig. 6**：$(\alpha_J,\alpha_W)$ learning phase diagram；展示 tutor 与 recurrence 的强度平衡。
- **Fig. 7**：uniform speed current 引起的 $U,V$ baseline shifts 满足 $\epsilon_u+\epsilon_v=1$ 与 $\epsilon_v=m\epsilon_u$。
- **Fig. 8**：baseline shift $\to$ wi\mathrm{d}th modulation $\to$ speed modulation，最终完成 time-varying unidirectional PI。
- **Fig. 9（Appendix）**：原理论 intrinsic-speed formula 与 simulation 之间的 $0.72$ 经验校正。

建议优先阅读顺序：

$$
\text{Eq. 1--6}
\rightarrow
\text{Eq. 10--13}
\rightarrow
\text{Eq. 18--22}
\rightarrow
\text{Eq. 23--26}
\rightarrow
\text{Eq. 27--36}.
$$

# 物理量对照表

| 出现顺序 | 物理量 | 维度 / 取值范围 | 含义 |
|---:|---|---|---|
| 1 | $L$ | 标量，长度 | 一维环境/内部流形长度 |
| 2 | $x,x^{\prime}$ | $[0,L)$ | 连续 neuron coordinate；在连接中通常分别表示 post/pre 坐标 |
| 3 | $z(t)$ | $[0,L)$ | moving tutor 或 bump center |
| 4 | $v(t)$ | 长度/时间 | 外部速度；训练前馈结构时通常为正常数 |
| 5 | $N_{\mathrm{in}}$ | 正整数 | input/tutor neurons 数量 |
| 6 | $N_c$ | 正整数 | competitive neurons 数量 |
| 7 | $\rho_{\mathrm{in}}$ | $N_{\mathrm{in}}/L$ | input neuron density |
| 8 | $\rho_c$ | $N_c/L$ | competitive neuron density |
| 9 | $\mathcal N(x;\mu,\sigma)$ | $1/\mathrm{length}$ | 积分为 1 的 Gaussian |
| 10 | $A_R$ | activity × length | tutor profile 的 integrated drive |
| 11 | $\sigma_R$ | 长度 | input tuning wi\mathrm{d}th |
| 12 | $\vec R(t)$ | $\mathbb R^{N_{\mathrm{in}}}$ | input-layer activity vector |
| 13 | $\vec U(t)$ | $\mathbb R^{N_c}$ | competitive membrane-potential vector |
| 14 | $\vec V(t)$ | $\mathbb R^{N_c}$ | firing-rate adaptation vector |
| 15 | $\vec r(t)$ | $\mathbb R_{\ge0}^{N_c}$ | competitive firing-rate vector |
| 16 | $\vec I(t)$ | $\mathbb R^{N_c}$ | competitive total input current |
| 17 | $\tau$ | 时间 | membrane time constant |
| 18 | $\tau_v$ | 时间 | adaptation time constant |
| 19 | $m$ | 无量纲/依 convention | adaptation strength |
| 20 | $k$ | 依 normalization | global divisive inhibition strength |
| 21 | $B(t)$ | 标量 | 全局 normalization factor |
| 22 | $\mathbf J$ | $\mathbb R_{\ge0}^{N_c\times N_{\mathrm{in}}}$ | feedforward weight matrix |
| 23 | $J_{ij}$ | 标量 | input $j	o$ competitive $i$ 的权重 |
| 24 | $\eta_J$ | $1/\mathrm{time}$ | feedforward learning rate |
| 25 | $\alpha_J$ | 依 weight units | feedforward local decay strength |
| 26 | $\beta$ | $(0,2)$ | weight-decay exponent；控制 equilibrium wi\mathrm{d}th |
| 27 | $A_J$ | weight amplitude | equilibrium Gaussian feedforward amplitude |
| 28 | $\sigma_J$ | 长度 | equilibrium feedforward wi\mathrm{d}th |
| 29 | $A_u,A_v,A_r$ | 各变量幅值 | $U,V,r$ Gaussian ansatz amplitudes |
| 30 | $\sigma_u$ | 长度 | $U,V$ bump wi\mathrm{d}th；$r$ wi\mathrm{d}th 为 $\sigma_u/\sqrt2$ |
| 31 | $d$ | 长度 | adaptation bump 相对 $U$ 的空间 lag |
| 32 | $\gamma$ | 无量纲 | first Hermite coefficient / bump asymmetry |
| 33 | $\Delta z_{\mathrm{pred}}$ | 长度 | prospective shift，约为 $\sigma_u\gamma$ |
| 34 | $u$ | 无量纲 | $u=\tau v/(\sqrt2\sigma_u)$ |
| 35 | $y$ | 无量纲 | $y=d/(\sqrt2\sigma_u)$ |
| 36 | $\Gamma$ | 无量纲 | $\Gamma=\tau_v/\tau$ |
| 37 | $\mathbf W$ | $\mathbb R_{\ge0}^{N_c\times N_c}$ | recurrent weight matrix |
| 38 | $W_{ij}$ | 标量 | competitive $j	o i$ 的 recurrent weight |
| 39 | $\eta_W$ | $1/\mathrm{time}$ | recurrent learning rate |
| 40 | $\alpha_W$ | 依 weight units | recurrent local decay strength |
| 41 | $A_W$ | weight amplitude | equilibrium recurrent Gaussian amplitude |
| 42 | $\sigma_W$ | 长度 | equilibrium recurrent wi\mathrm{d}th |
| 43 | $v_{\mathrm{int}}(m)$ | 长度/时间 | 撤去 tutor 后 moving bump 的 intrinsic speed |
| 44 | $g$ | current / speed | speed-error 到 uniform current 的 gain |
| 45 | $I_{\mathrm{speed}}$ | current | $g[v(t)-v_{\mathrm{int}}]$ |
| 46 | $\epsilon_u,\epsilon_v$ | 无量纲 | speed current 在 $U,V$ baseline 中的分配系数 |
| 47 | $\lambda_n$ | $1/\mathrm{time}$ | weight-perturbation mode 的增长率 |
| 48 | $\hat L$ | 线性算符 | feedforward weight stability operator |